# Chapter 17 — Machine Learning with Scikit-learn

Scikit-learn provides a consistent API for preprocessing, model fitting, evaluation, and model selection. Learn the end-to-end supervised-learning workflow while avoiding leakage and overclaiming.

## Prerequisites
Python, NumPy, Pandas, descriptive statistics, and basic train/test reasoning.

## Learning goals
- Define features, target, task type, and a baseline.
- Split data before fitting transformations.
- Build preprocessing and estimator pipelines.
- Train regression and classification models.
- Evaluate with task-appropriate metrics and cross-validation.
- Recognize leakage, overfitting, and reproducibility requirements.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. Estimator API
2. Splitting and leakage
3. Preprocessing and pipelines
4. Regression and classification
5. Evaluation metrics
6. Cross-validation and tuning

### Estimator API
Most estimators follow `fit(X, y)`, `predict(X)`, and sometimes `predict_proba(X)`. Transformers expose `fit`, `transform`, and `fit_transform`. `get_params()` inspects hyperparameters; `set_params()` changes them. Inputs are commonly arrays or DataFrames with rows as samples and columns as features; preserve feature names where possible.

#### Demonstration 1 — Build a regression baseline

Run the code and inspect both the output and the assumptions behind it.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sales=pd.read_csv(DATA/"sales.csv")
X=sales[["units","discount"]]; y=sales["revenue"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
model=DummyRegressor(strategy="mean").fit(X_train,y_train); pred=model.predict(X_test)
print("MAE",mean_absolute_error(y_test,pred),"RMSE",mean_squared_error(y_test,pred)**.5,"R2",r2_score(y_test,pred))

Data directory: /mnt/data/python_data_analysis_book/data
MAE 179953.2602666667 RMSE 263125.3451224286 R2 -0.018181211861579172


### Splitting and leakage
`train_test_split(X,y,test_size,train_size,random_state,shuffle,stratify)` separates data. Split before fitting imputation, scaling, feature selection, or encoders. Leakage occurs when information unavailable at prediction time enters training, including preprocessing fitted on all rows and future time information. For grouped or temporal data use group/time-aware splitters, not random splitting.

#### Demonstration 2 — Create a preprocessing pipeline

Run the code and inspect both the output and the assumptions behind it.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
num_cols=["units","discount"]; cat_cols=["region","product"]
pre=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_cols),("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)])
pipe=Pipeline([("preprocess",pre),("model",LinearRegression())])
X=sales[num_cols+cat_cols]; y=sales.revenue
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
pipe.fit(X_train,y_train); pred=pipe.predict(X_test); print("MAE:",mean_absolute_error(y_test,pred),"R2:",r2_score(y_test,pred))

MAE: 75331.49631673291 R2: 0.8069030879182847


### Preprocessing and pipelines
`SimpleImputer(strategy=...)` fills missing values; `StandardScaler` centers/scales; `OneHotEncoder(handle_unknown="ignore")` encodes categories. `ColumnTransformer` applies transformations to subsets of columns. `Pipeline(steps=[...])` chains steps and fits them inside each validation fold, reducing leakage risk. Sparse/dense output depends on encoder configuration and data.

#### Demonstration 3 — Train a classification pipeline

Run the code and inspect both the output and the assumptions behind it.

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
data=load_breast_cancer(as_frame=True); Xtr,Xte,ytr,yte=train_test_split(data.data,data.target,test_size=.25,random_state=42,stratify=data.target)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)); clf.fit(Xtr,ytr); pred=clf.predict(Xte)
print(confusion_matrix(yte,pred)); print(classification_report(yte,pred,target_names=data.target_names))

[[52  1]
 [ 1 89]]
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        53
      benign       0.99      0.99      0.99        90

    accuracy                           0.99       143
   macro avg       0.99      0.99      0.99       143
weighted avg       0.99      0.99      0.99       143



### Regression and classification
Regression targets are numeric; common baselines include `DummyRegressor`, `LinearRegression`, and tree-based regressors. Classification predicts classes; models include `LogisticRegression`, decision trees, random forests, and support vector machines. Classification probabilities require calibration checks before high-stakes threshold decisions.

#### Demonstration 4 — Use cross-validation and parameter search

Run the code and inspect both the output and the assumptions behind it.

In [5]:
from sklearn.model_selection import cross_val_score,GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
rf=RandomForestRegressor(random_state=42)
scores=cross_val_score(rf,X_train.assign(region="all",product="all")[["units","discount"]],y_train,cv=5,scoring="neg_mean_absolute_error")
print("CV MAE:",-scores.mean())
search=GridSearchCV(RandomForestRegressor(random_state=42),{"n_estimators":[100,200],"max_depth":[None,5]},cv=5,scoring="neg_mean_absolute_error",n_jobs=-1)
search.fit(X_train[["units","discount"]],y_train); print(search.best_params_, -search.best_score_)

CV MAE: 203534.62369122988
{'max_depth': 5, 'n_estimators': 100} 197900.49845163565


### Evaluation metrics
Regression: MAE, MSE/RMSE, and R-squared; interpret units and outlier sensitivity. Classification: confusion matrix, accuracy, precision, recall, F1, ROC-AUC, and PR-AUC. Metrics depend on class balance and decision thresholds. Always compare with a baseline and report the data split and evaluation sample.

### Cross-validation and tuning
`cross_val_score` estimates performance across folds. `GridSearchCV` and `RandomizedSearchCV` tune parameters; their search must be conducted within training data, leaving a final test set untouched. `scoring` determines the metric; `cv` determines fold strategy. Set random seeds where applicable and record versions.

## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: Estimator API.
- Explain one important concept from: Splitting and leakage.
- Explain one important concept from: Preprocessing and pipelines.
- Explain one important concept from: Regression and classification.
- Explain one important concept from: Evaluation metrics.